In [47]:
import pandas as pd
import sqlite3

In [48]:
orders = pd.read_csv("orders.csv")
orders.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [49]:
users = pd.read_json("users.json")
users.head()


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [50]:
conn = sqlite3.connect("restaurants.db")
cursor = conn.cursor()


In [51]:
file = open("restaurants.sql", "r")
sql_script = file.read()
file.close()


In [52]:
cursor.executescript(sql_script)
conn.commit()


OperationalError: table restaurants already exists

In [53]:
restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
restaurants.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [54]:
orders_users = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)

orders_users.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


In [55]:
final_df = pd.merge(
    orders_users,
    restaurants,
    on="restaurant_id",
    how="left"
)

final_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [56]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [57]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [60]:
gold_df = final_df[final_df["membership"] == "Gold"]
gold_orders_count = gold_df.shape[0]
gold_orders_count


4987

In [61]:
gold_df = final_df[final_df["membership"] == "Gold"]

city_revenue = gold_df.groupby("city")["total_amount"].sum()

city_revenue


,total_amount
city,
Bangalore,994702.59
Chennai,1080909.79
Hyderabad,896740.19
Pune,1003012.32


In [62]:
city_revenue.idxmax()


'Chennai'

In [63]:
cuisine_avg = final_df.groupby("cuisine")["total_amount"].mean()

cuisine_avg


,total_amount
cuisine,
Chinese,798.389020
Indian,798.466011
Italian,799.448578
Mexican,808.021344


In [64]:
cuisine_avg.idxmax()


'Mexican'

In [65]:
user_total = final_df.groupby("user_id")["total_amount"].sum()

user_total


,total_amount
user_id,
1,1289.66
2,7564.12
3,1839.51
4,3741.16
5,5742.88
...,...
2996,1533.54
2997,5310.32
2998,4241.47


In [66]:
count_users = (user_total > 1000).sum()

count_users


np.int64(2544)

In [68]:
final_df["rating_range"] = pd.cut(
    final_df["rating"],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [69]:
rating_revenue = final_df.groupby("rating_range")["total_amount"].sum()

rating_revenue


/tmp/ipython-input-825580444.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rating_revenue = final_df.groupby("rating_range")["total_amount"].sum()


,total_amount
rating_range,
3.0–3.5,2136772.70
3.6–4.0,1717494.41
4.1–4.5,1960326.26
4.6–5.0,2197030.75


In [70]:
rating_revenue.idxmax()


'4.6–5.0'

In [71]:
gold_df = final_df[final_df["membership"] == "Gold"]

gold_city_avg = gold_df.groupby("city")["total_amount"].mean()

gold_city_avg


,total_amount
city,
Bangalore,793.223756
Chennai,808.459080
Hyderabad,806.421034
Pune,781.162243


In [72]:
gold_city_avg.idxmax()


'Chennai'

In [73]:
restaurant_count = final_df.groupby("cuisine")["restaurant_id"].nunique()
restaurant_count


,restaurant_id
cuisine,
Chinese,120
Indian,126
Italian,126
Mexican,128


In [74]:
cuisine_revenue = final_df.groupby("cuisine")["total_amount"].sum()
cuisine_revenue


,total_amount
cuisine,
Chinese,1930504.65
Indian,1971412.58
Italian,2024203.80
Mexican,2085503.09


In [75]:
total_orders = final_df.shape[0]

gold_orders = final_df[final_df["membership"] == "Gold"].shape[0]

percentage = (gold_orders / total_orders) * 100

round(percentage)


50

In [76]:
restaurant_avg = final_df.groupby("restaurant_name_y")["total_amount"].mean()
restaurant_count = final_df.groupby("restaurant_name_y")["order_id"].count()


In [77]:
restaurant_summary = pd.DataFrame()
restaurant_summary["avg_order_value"] = restaurant_avg
restaurant_summary["order_count"] = restaurant_count


In [78]:
filtered = restaurant_summary[restaurant_summary["order_count"] < 20]

filtered.sort_values("avg_order_value", ascending=False).head()


,avg_order_value,order_count
restaurant_name_y,,
Restaurant_294,1040.222308,13
Restaurant_262,1029.473333,18
Restaurant_77,1029.180833,12
Restaurant_193,1026.306667,15
Restaurant_7,1002.140625,16


In [79]:
restaurant_info = final_df[
    ["restaurant_name_y", "cuisine"]
].drop_duplicates()


In [80]:
restaurant_info


,restaurant_name_y,cuisine
0,Restaurant_450,Mexican
1,Restaurant_309,Indian
2,Restaurant_107,Mexican
3,Restaurant_224,Chinese
4,Restaurant_293,Italian
...,...,...
2098,Restaurant_219,Mexican
2135,Restaurant_337,Mexican
2198,Restaurant_29,Italian
2552,Restaurant_480,Chinese


In [81]:
fixed_table = restaurant_info.merge(
    filtered,
    left_on="restaurant_name_y",
    right_index=True
)


In [82]:
fixed_table.sort_values("avg_order_value", ascending=False)


,restaurant_name_y,cuisine,avg_order_value,order_count
1407,Restaurant_294,Italian,1040.222308,13
52,Restaurant_262,Chinese,1029.473333,18
1389,Restaurant_77,Mexican,1029.180833,12
1720,Restaurant_193,Chinese,1026.306667,15
8,Restaurant_7,Italian,1002.140625,16
...,...,...,...,...
114,Restaurant_184,Mexican,621.828947,19
831,Restaurant_498,Chinese,596.815556,18
1167,Restaurant_192,Italian,589.972857,14
236,Restaurant_329,Chinese,578.578667,15


In [83]:
combo_revenue = final_df.groupby(
    ["membership", "cuisine"]
)["total_amount"].sum()

combo_revenue


membership  cuisine
Gold        Chinese     977713.74
            Indian      979312.31
            Italian    1005779.05
            Mexican    1012559.79
Regular     Chinese     952790.91
            Indian      992100.27
            Italian    1018424.75
            Mexican    1072943.30
Name: total_amount, dtype: float64

In [84]:
combo_revenue.idxmax()


('Regular', 'Mexican')

In [85]:
final_df["order_date"] = pd.to_datetime(
    final_df["order_date"],
    dayfirst=True
)

final_df["quarter"] = final_df["order_date"].dt.quarter


In [89]:
quarter_revenue = final_df.groupby("quarter")["total_amount"].sum()

quarter_revenue

,total_amount
quarter,
1,2010626.64
2,1945348.72
3,2037385.10
4,2018263.66


In [96]:
gold_orders = final_df[final_df["membership"] == "Gold"]

total_gold_orders = len(gold_orders)

total_gold_orders


4987

In [97]:
hyd_df = final_df[final_df["city"] == "Hyderabad"]

hyd_revenue = hyd_df["total_amount"].sum()

round(hyd_revenue)


1889367

In [98]:
users = final_df["user_id"]

unique_users = users.unique()

len(unique_users)


2883

In [99]:
gold_df = final_df[final_df["membership"] == "Gold"]

gold_amounts = gold_df["total_amount"]

avg_gold_order = gold_amounts.mean()

round(avg_gold_order, 2)


np.float64(797.15)

In [100]:
high_rating_df = final_df[final_df["rating"] >= 4.5]

len(high_rating_df)


3374

In [101]:
gold_df = final_df[final_df["membership"] == "Gold"]


In [102]:
city_revenue = gold_df.groupby("city")["total_amount"].sum()

city_revenue


,total_amount
city,
Bangalore,994702.59
Chennai,1080909.79
Hyderabad,896740.19
Pune,1003012.32


In [104]:
top_city_orders = gold_df[gold_df["city"] == top_gold_city]

len(top_city_orders)


1337

In [105]:
final_df.shape[0]

10000